# 01 — Cache residual-stream activations

Runs Llama 3.1 8B forward passes over SelfAware prompts and saves `resid_post` at every layer, plus the generated responses for the judge.

**Inputs:** `data/selfaware/SelfAware.json` (topic-enriched).
**Outputs:** `responses.json`, `acts_layer_{i}.pt` (one per layer) in the results dir.

**Heavy step.** Run on Colab A100. `results_dir()` resolves to Google Drive on Colab, so caching survives a runtime restart — re-running this notebook skips already-cached shards.

In [1]:
# --- Colab setup (skip these two lines when running locally) ---
from google.colab import drive; drive.mount('/content/drive')
%cd "/content/drive/Othercomputers/My MacBook Air/abstention-geometry"
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Othercomputers/My MacBook Air/abstention-geometry


In [2]:
import json

from src.data import load_selfaware, build_prompts
from src.model import load_model, generate_responses, cache_residual_stream
from src.paths import results_dir

In [4]:
import src

# Resolve paths from the src package location, not the working directory —
# robust whether or not the Colab %cd above succeeded.
REPO_ROOT = Path(src.__file__).resolve().parent.parent
DATA_PATH = str(REPO_ROOT / 'data' / 'selfaware' / 'SelfAware.json')
RESULTS_DIR = results_dir()  # Drive on Colab, repo results/ locally; override with $RESULTS_DIR
print('DATA_PATH   =', DATA_PATH)
print('RESULTS_DIR =', RESULTS_DIR)

DATA_PATH   = /content/drive/Othercomputers/My MacBook Air/abstention-geometry/data/selfaware/SelfAware.json
RESULTS_DIR = /content/drive/Othercomputers/My MacBook Air/abstention-geometry/results/


## Load dataset + build prompts

In [5]:
df = load_selfaware(DATA_PATH)
prompts = build_prompts(df)
print(f'{len(df)} questions | answerable rate: {df.answerable.mean():.2f}')
print(df.topic.value_counts())
print('\n--- example prompt ---\n' + prompts[0])

3369 questions | answerable rate: 0.69
topic
uncategorized    3369
Name: count, dtype: int64

--- example prompt ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Answer the question if you can. If you do not know or are not sure, say so clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>

What form of entertainment are 'Slow Poke' and 'You Belong to Me'?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## Load model

In [6]:
model = load_model("meta-llama/Llama-3.1-8B-Instruct")
print('loaded', model.cfg.model_name, '|', model.cfg.n_layers, 'layers | d_model', model.cfg.d_model)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.1-8B-Instruct into HookedTransformer
loaded Llama-3.1-8B-Instruct | 32 layers | d_model 4096


## Generate responses

Greedy decoding. Save to `responses.json` for the judge in 02.

In [ ]:
responses = generate_responses(model, prompts, batch_size=8)
with open(RESULTS_DIR + 'responses.json', 'w') as f:
    json.dump({'question_id': df['question_id'].tolist(), 'response': responses}, f)
print('saved', RESULTS_DIR + 'responses.json')
print('\n--- example response ---\n' + responses[0])

generating:  50%|█████     | 212/422 [1:07:16<1:06:34, 19.02s/it]

## Cache residual stream

All 32 layers, last-token resid_post. Shard-checkpointed: re-running after a disconnect resumes from the last completed shard.

In [ ]:
cache_residual_stream(model, prompts, batch_size=8, save_dir=RESULTS_DIR)
print('done — acts_layer_0..%d.pt written to %s' % (model.cfg.n_layers - 1, RESULTS_DIR))